<a href="https://colab.research.google.com/github/sahibbedi/signal-scanner/blob/main/signal_scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import requests
import pandas as pd
import time
import matplotlib.pyplot as plt

# --- Configuration ---
HEADERS = {'User-Agent': 'University of Illinois Research Project (sahibb2@illinois.edu)'}

# Default Matrix (Fallback if the user just presses Enter)
DEFAULT_FUNDS = {
    "0001273087": "Millennium Management",
    "0001446194": "Susquehanna International Group",
    "0000854157": "State of Wisconsin Investment Board",
    "0001037389": "Renaissance Technologies",
    "0001067983": "Berkshire Hathaway"
}

BTC_ETFS_8 = ['09652W10', '31635V10', '74347G44', '05970V10', '38963710']

# Weighted Scoring Engine
PRIMARY_KEYWORDS = ['digital asset', 'cryptocurrency', 'bitcoin']
SECONDARY_KEYWORDS = ['blockchain', 'virtual currency', 'crypto']
ETF_EXPOSURE_POINTS = 60

def analyze_fund(cik, name=None):
    cik_padded = cik.zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"

    try:
        response = requests.get(url, headers=HEADERS)
        response.raise_for_status()
        data = response.json()

        # Dynamically grab the official SEC name if the user only provided a CIK
        if not name:
            name = data.get('name', f"Unknown Entity (CIK {cik})")

        print(f"  > Scanning {name} (CIK: {cik_padded})...")
    except requests.exceptions.RequestException:
        print(f"  [!] Rate limited or invalid CIK ({cik_padded}). Try again later.")
        return None

    recent_filings = pd.DataFrame(data['filings']['recent'])

    signal_score = 0
    has_13f_signal = False
    latest_13f_date = "None"

    if not recent_filings.empty:
        # 1. 13F Check
        forms_13f = recent_filings[recent_filings['form'] == '13F-HR']
        if not forms_13f.empty:
            latest_13f_date = forms_13f.iloc[0]['filingDate']
            accession_number = forms_13f.iloc[0]['accessionNumber']
            accession_no_dashes = accession_number.replace('-', '')

            txt_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession_no_dashes}/{accession_number}.txt"
            try:
                txt_resp = requests.get(txt_url, headers=HEADERS)
                if txt_resp.status_code == 200:
                    txt_content = txt_resp.text
                    has_13f_signal = any(cusip in txt_content for cusip in BTC_ETFS_8)
                    if has_13f_signal:
                        signal_score += ETF_EXPOSURE_POINTS
            except Exception:
                pass

        # 2. Deep-scan the raw text
        for index, row in recent_filings.head(3).iterrows():
            if row['form'] != '13F-HR':
                acc_num = row['accessionNumber']
                acc_no_dashes = acc_num.replace('-', '')
                doc_url = f"https://www.sec.gov/Archives/edgar/data/{int(cik)}/{acc_no_dashes}/{acc_num}.txt"

                try:
                    doc_resp = requests.get(doc_url, headers=HEADERS)
                    if doc_resp.status_code == 200:
                        doc_content = doc_resp.text.lower()
                        if any(keyword in doc_content for keyword in PRIMARY_KEYWORDS):
                            signal_score += 20
                        if any(keyword in doc_content for keyword in SECONDARY_KEYWORDS):
                            signal_score += 10
                except:
                    pass
                time.sleep(0.15)

    # 3. Categorize Lead Tier
    if signal_score >= 60:
        tier = "🔥 Tier 1 (Active Allocator)"
    elif signal_score >= 20:
        tier = "🟡 Tier 2 (IPS Flexible)"
    else:
        tier = "❄️ Tier 3 (Cold/Blocked)"

    time.sleep(0.15)

    return {
        "Fund Name": name,
        "CIK": cik_padded,
        "Signal Score": signal_score,
        "Tier": tier,
        "BTC ETF Exposure": "Yes" if has_13f_signal else "No",
        "Latest 13F": latest_13f_date
    }

def visualize_pipeline(df):
    if len(df) <= 1:
        return # Skip chart if we only looked up one CIK

    print("\nGenerating Pipeline Visualization...")
    df_sorted = df.sort_values(by='Signal Score', ascending=True)
    colors = ['#E63946' if score >= 60 else '#F4A261' if score >= 20 else '#457B9D' for score in df_sorted['Signal Score']]

    plt.figure(figsize=(10, max(4, len(df)*0.8)))
    plt.barh(df_sorted['Fund Name'], df_sorted['Signal Score'], color=colors)
    plt.title('Institutional Crypto Intent: Lead Ranking', fontsize=14, fontweight='bold')
    plt.xlabel('Algorithmic Signal Score', fontweight='bold')
    plt.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# --- Execution ---
if __name__ == "__main__":
    print("="*85)
    print(" 🔎 INSTITUTIONAL SIGNAL SCANNER (CIK MODE)")
    print("="*85)

    print("\nOptions:")
    print("1. Type a specific SEC CIK Number (e.g., 0001364742 for BlackRock) and press Enter.")
    print("2. Leave blank and press Enter to scan the Default Institutional Matrix.")
    user_input = input("\nEnter CIK: ").strip()

    results = []

    if user_input:
        # Clean input to ensure only numbers are processed
        clean_cik = ''.join(filter(str.isdigit, user_input))
        if clean_cik:
            print("\nInitializing Search...")
            profile = analyze_fund(clean_cik)
            if profile:
                results.append(profile)
        else:
            print("\n[!] Invalid CIK format. Please enter numbers only.")
    else:
        print("\nRunning default matrix...")
        for cik, name in DEFAULT_FUNDS.items():
            profile = analyze_fund(cik, name)
            if profile:
                results.append(profile)

    if results:
        intel_report = pd.DataFrame(results)
        print("\n" + "="*85)
        print(" 🎯 FUND INTELLIGENCE REPORT")
        print("="*85)
        print(intel_report.to_string(index=False))
        visualize_pipeline(intel_report)

 🔎 INSTITUTIONAL SIGNAL SCANNER (CIK MODE)

Options:
1. Type a specific SEC CIK Number (e.g., 0001364742 for BlackRock) and press Enter.
2. Leave blank and press Enter to scan the Default Institutional Matrix.

Enter CIK: 0001273087

Initializing Search...
  > Scanning MILLENNIUM MANAGEMENT LLC (CIK: 0001273087)...

 🎯 FUND INTELLIGENCE REPORT
                Fund Name        CIK  Signal Score                        Tier BTC ETF Exposure Latest 13F
MILLENNIUM MANAGEMENT LLC 0001273087            60 🔥 Tier 1 (Active Allocator)              Yes 2026-05-15
